# ChemXAI Toolbox - Testes de Explicabilidade

Este notebook demonstra o uso do toolbox ChemXAI para:
1. **Teste MLP + QM9**: Previsão de todas as 15 propriedades do QM9 usando descritores Physicochemical com explicações SHAP e LIME
2. **Teste GCN + PCQM4**: Previsão HOMO-LUMO com GCN e explicações usando GNNExplainer, GraphSHAP e GraphLIME

---

In [ ]:
# Imports e Configurações Iniciais
import sys
import os

# Adicionar o diretório raiz ao path
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import Optuna para otimização de hiperparâmetros
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Imports do ChemXAI
from chemxai.data import qm9_tabular, graph_datasets
from chemxai.models import MLP, GCN
from chemxai.explainers import Shap, LIME, GNNExplain, GraphShap, GraphLIME
from chemxai.evaluate import TabularAnalyzer
from chemxai.plots import horizontal_bar_plot, radar_plot

# Configuração do dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Criar diretórios para salvar resultados
results_dir = os.path.join(os.getcwd(), 'explanation_results')
os.makedirs(results_dir, exist_ok=True)
os.makedirs(os.path.join(results_dir, 'mlp_qm9'), exist_ok=True)
os.makedirs(os.path.join(results_dir, 'gcn_pcqm4'), exist_ok=True)
os.makedirs(os.path.join(results_dir, 'optuna_studies'), exist_ok=True)
print(f"Diretórios de resultados criados em: {results_dir}")

## Parte 1: MLP + QM9 com Descritores Physicochemical

### 1.1 Função para Visualizar Arquitetura do Modelo

In [ ]:
def visualize_mlp_architecture(model, input_dim, output_dim, save_path=None, title="MLP Architecture"):
    """
    Cria uma visualização simplificada da arquitetura do modelo MLP.
    
    Parameters:
    -----------
    model : MLP
        Modelo MLP a ser visualizado
    input_dim : int
        Dimensão da entrada
    output_dim : int
        Dimensão da saída
    save_path : str, optional
        Caminho para salvar a imagem
    title : str
        Título do gráfico
    """
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    
    # Extrair dimensões das camadas do modelo
    layer_dims = [input_dim]
    for name, layer in model.layers.named_children():
        if isinstance(layer, nn.Linear):
            layer_dims.append(layer.out_features)
    
    n_layers = len(layer_dims)
    x_positions = np.linspace(1, 9, n_layers)
    
    # Configurações visuais
    max_neurons_display = 8  # Máximo de neurônios exibidos por camada
    neuron_radius = 0.15
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, n_layers))
    
    for i, (x, dim) in enumerate(zip(x_positions, layer_dims)):
        n_display = min(dim, max_neurons_display)
        y_positions = np.linspace(2, 8, n_display)
        
        # Desenhar neurônios
        for y in y_positions:
            circle = plt.Circle((x, y), neuron_radius, color=colors[i], ec='black', linewidth=1.5)
            ax.add_patch(circle)
        
        # Se há mais neurônios que o exibido, mostrar reticências
        if dim > max_neurons_display:
            ax.text(x, 5, '...', fontsize=20, ha='center', va='center', fontweight='bold')
        
        # Label da camada
        if i == 0:
            label = f'Input\n({dim})'
        elif i == n_layers - 1:
            label = f'Output\n({dim})'
        else:
            label = f'Hidden {i}\n({dim})'
        ax.text(x, 0.8, label, ha='center', va='center', fontsize=11, fontweight='bold')
        
        # Desenhar conexões para próxima camada
        if i < n_layers - 1:
            next_x = x_positions[i + 1]
            next_dim = layer_dims[i + 1]
            next_n_display = min(next_dim, max_neurons_display)
            next_y_positions = np.linspace(2, 8, next_n_display)
            
            # Desenhar algumas conexões (não todas para não poluir)
            for j, y1 in enumerate(y_positions[::2]):  # Apenas metade das conexões
                for k, y2 in enumerate(next_y_positions[::2]):
                    alpha = 0.1 + 0.1 * (1 - abs(j - k) / max(len(y_positions), len(next_y_positions)))
                    ax.plot([x + neuron_radius, next_x - neuron_radius], [y1, y2], 
                           'gray', alpha=alpha, linewidth=0.5)
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Adicionar legenda com informações do modelo
    info_text = f"Layers: {layer_dims}\nActivation: ReLU\nTotal Parameters: {sum(p.numel() for p in model.parameters()):,}"
    ax.text(5, 9.5, info_text, ha='center', va='top', fontsize=10, 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Arquitetura salva em: {save_path}")
    
    plt.show()
    return fig

### 1.2 Carregamento dos Dados QM9 e Otimização com Optuna

In [ ]:
# Função objetivo do Optuna para otimização da MLP
def create_mlp_objective(train_loader, val_loader, input_dim, device):
    """
    Cria a função objetivo para otimização de hiperparâmetros da MLP com Optuna.
    """
    def objective(trial):
        # Sugerir hiperparâmetros
        n_layers = trial.suggest_int('n_layers', 1, 4)
        layers = []
        for i in range(n_layers):
            layer_size = trial.suggest_int(f'layer_{i}_size', 32, 256, step=32)
            layers.append(layer_size)
        
        lr = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
        dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5, step=0.1)
        
        # Criar modelo com os hiperparâmetros sugeridos
        class MLPWithDropout(nn.Module):
            def __init__(self, input_dim, output_dim, layers, dropout_rate):
                super(MLPWithDropout, self).__init__()
                all_layers = []
                prev_dim = input_dim
                
                for layer_dim in layers:
                    all_layers.append(nn.Linear(prev_dim, layer_dim))
                    all_layers.append(nn.ReLU())
                    all_layers.append(nn.Dropout(dropout_rate))
                    prev_dim = layer_dim
                
                all_layers.append(nn.Linear(prev_dim, output_dim))
                self.layers = nn.Sequential(*all_layers)
            
            def forward(self, x):
                return self.layers(x)
        
        model = MLPWithDropout(input_dim, 1, layers, dropout_rate).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = nn.L1Loss()
        
        # Treinar por algumas épocas
        n_epochs = 20
        for epoch in range(n_epochs):
            model.train()
            for batch_x, batch_y in train_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
            
            # Validação
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_x, batch_y in val_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    outputs = model(batch_x)
                    val_loss += criterion(outputs, batch_y).item()
            
            val_loss = val_loss / len(val_loader)
            
            # Reportar valor intermediário para pruning
            trial.report(val_loss, epoch)
            
            # Pruning: interromper trials ruins
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        
        return val_loss
    
    return objective


def optimize_mlp_hyperparameters(qm9, att_index, device, n_trials=50, timeout=600):
    """
    Otimiza hiperparâmetros da MLP usando Optuna.
    
    Parameters:
    -----------
    qm9 : qm9_tabular
        Objeto com os dados do QM9
    att_index : int
        Índice da propriedade a ser predita
    device : torch.device
        Dispositivo para treinamento
    n_trials : int
        Número de trials do Optuna
    timeout : int
        Tempo máximo em segundos
        
    Returns:
    --------
    study : optuna.Study
        Estudo do Optuna com os resultados
    best_params : dict
        Melhores hiperparâmetros encontrados
    """
    print(f"\n{'='*60}")
    print(f"Otimizando hiperparâmetros MLP para propriedade {att_index}")
    print(f"{'='*60}")
    
    # Carregar dados
    train_loader, val_loader, test_loader, _, _, _, _ = qm9.get_paired_dataloaders_tabular(
        att_index=att_index,
        batch_size=64,
        descriptor_type='Physicochemical',
        n_noise=0
    )
    
    # Obter dimensão de entrada
    sample_batch = next(iter(train_loader))
    input_dim = sample_batch[0].shape[1]
    
    # Criar estudo Optuna
    study = optuna.create_study(
        direction='minimize',
        study_name=f'mlp_qm9_att{att_index}',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    )
    
    # Criar função objetivo
    objective = create_mlp_objective(train_loader, val_loader, input_dim, device)
    
    # Otimizar
    study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=True)
    
    print(f"\nMelhores hiperparâmetros encontrados:")
    print(f"  Validation MAE: {study.best_value:.4f}")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study, study.best_params, input_dim

In [ ]:
# Executar otimização para uma propriedade exemplo (HOMO energy - índice 5)
# Para todas as propriedades, iterar sobre a lista
example_att_index = 5  # HOMO energy

# Carregar dados primeiro
qm9 = qm9_tabular()

# Executar otimização Optuna
mlp_study, mlp_best_params, input_dim = optimize_mlp_hyperparameters(
    qm9=qm9,
    att_index=example_att_index,
    device=device,
    n_trials=30,  # Aumentar para resultados melhores
    timeout=300   # 5 minutos por propriedade
)

# Visualizar resultados da otimização
fig_history = optuna.visualization.plot_optimization_history(mlp_study)
fig_history.write_image(os.path.join(results_dir, 'optuna_studies', 'mlp_optimization_history.png'))
fig_history.show()

fig_importance = optuna.visualization.plot_param_importances(mlp_study)
fig_importance.write_image(os.path.join(results_dir, 'optuna_studies', 'mlp_param_importances.png'))
fig_importance.show()

print(f"\nGráficos de otimização salvos em: {os.path.join(results_dir, 'optuna_studies')}")

In [ ]:
# Carregar dados QM9 com descritores Physicochemical
qm9 = qm9_tabular()

# Propriedades do QM9
property_names = [
    'Rotational constant A (GHz)', 'Rotational constant B (GHz)', 'Rotational constant C (GHz)',
    'Dipole moment (D)', 'Isotropic polarizability (a.u.)',
    'HOMO energy (Ha)', 'LUMO energy (Ha)', 'Gap energy (Ha)', 
    'Electronic spatial extent (a.u.)', 'Zero point vibrational energy (Ha)',
    'Internal energy at 0K (Ha)', 'Internal energy at 298K (Ha)',
    'Enthalpy at 298K (Ha)', 'Free energy at 298K (Ha)', 'Heat capacity at 298K (cal/mol·K)'
]

print(f"Número total de propriedades: {len(property_names)}")
print("\nPropriedades disponíveis:")
for i, prop in enumerate(property_names):
    print(f"  [{i}] {prop}")

### 1.3 Função para Treinar e Explicar MLP para Cada Propriedade

In [ ]:
def train_and_explain_mlp_optimized(qm9, att_index, property_name, device, results_dir, 
                                     best_params, epochs=50, n_samples_explain=100):
    """
    Treina um modelo MLP com hiperparâmetros otimizados e gera explicações SHAP e LIME.
    
    Parameters:
    -----------
    qm9 : qm9_tabular
        Objeto com os dados do QM9
    att_index : int
        Índice da propriedade a ser predita
    property_name : str
        Nome da propriedade
    device : torch.device
        Dispositivo para treinamento
    results_dir : str
        Diretório para salvar resultados
    best_params : dict
        Melhores hiperparâmetros do Optuna
    epochs : int
        Número de épocas de treinamento
    n_samples_explain : int
        Número de amostras para explicação
        
    Returns:
    --------
    dict : Dicionário com resultados do treinamento e explicações
    """
    print(f"\n{'='*60}")
    print(f"Treinando MLP OTIMIZADA para: {property_name} (att_index={att_index})")
    print(f"{'='*60}")
    
    # Extrair hiperparâmetros
    n_layers = best_params['n_layers']
    layers = [best_params.get(f'layer_{i}_size', 128) for i in range(n_layers)]
    lr = best_params['learning_rate']
    dropout_rate = best_params['dropout_rate']
    
    print(f"Hiperparâmetros: layers={layers}, lr={lr:.6f}, dropout={dropout_rate}")
    
    # Obter dataloaders
    batch_size = 64
    train_loader, val_loader, test_loader, _, _, _, _ = qm9.get_paired_dataloaders_tabular(
        att_index=att_index,
        batch_size=batch_size,
        descriptor_type='Physicochemical',
        n_noise=0
    )
    
    # Obter dimensão de entrada
    sample_batch = next(iter(train_loader))
    input_dim = sample_batch[0].shape[1]
    output_dim = 1
    
    print(f"Dimensão de entrada: {input_dim}")
    print(f"Amostras de treino: {len(train_loader.dataset)}")
    
    # Criar modelo MLP com dropout
    class MLPWithDropout(nn.Module):
        def __init__(self, input_dim, output_dim, layers, dropout_rate):
            super(MLPWithDropout, self).__init__()
            all_layers = []
            prev_dim = input_dim
            
            for layer_dim in layers:
                all_layers.append(nn.Linear(prev_dim, layer_dim))
                all_layers.append(nn.ReLU())
                all_layers.append(nn.Dropout(dropout_rate))
                prev_dim = layer_dim
            
            all_layers.append(nn.Linear(prev_dim, output_dim))
            self.layers = nn.Sequential(*all_layers)
        
        def forward(self, x):
            return self.layers(x)
    
    model = MLPWithDropout(input_dim, output_dim, layers, dropout_rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()
    
    # Treinar modelo
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        train_loss = epoch_loss / len(train_loader)
        train_losses.append(train_loss)
        
        # Validação
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                outputs = model(batch_x)
                val_loss += criterion(outputs, batch_y).item()
        
        val_loss = val_loss / len(val_loader)
        val_losses.append(val_loss)
        
        # Salvar melhor modelo
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] - Train: {train_loss:.4f} - Val: {val_loss:.4f}")
    
    # Carregar melhor modelo
    model.load_state_dict(best_model_state)
    
    # Avaliar no conjunto de teste
    model.eval()
    test_predictions = []
    test_targets = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            test_predictions.append(outputs.cpu())
            test_targets.append(batch_y.cpu())
    
    test_predictions = torch.cat(test_predictions)
    test_targets = torch.cat(test_targets)
    test_mae = torch.mean(torch.abs(test_predictions - test_targets)).item()
    print(f"✅ MAE no conjunto de teste: {test_mae:.4f}")
    
    # Preparar dados para explicação
    all_train_x = []
    for batch_x, _ in train_loader:
        all_train_x.append(batch_x)
    train_tensor = torch.cat(all_train_x)[:n_samples_explain]
    
    all_test_x = []
    for batch_x, _ in test_loader:
        all_test_x.append(batch_x)
    test_tensor = torch.cat(all_test_x)[:min(50, len(all_test_x[0]))]
    
    print(f"\nGerando explicações SHAP...")
    shap_explainer = Shap(model, train_tensor, test_tensor, device)
    shap_global = shap_explainer.explain_global()
    shap_local = shap_explainer.explain_local(0)
    
    print(f"Gerando explicações LIME...")
    lime_explainer = LIME(model, train_tensor, test_tensor, device, mode='regression')
    lime_local = lime_explainer.explain_local(0)
    
    # Nomes das features
    feature_names = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
                    "NumRotatableBonds", "NumAromaticRings", "BalabanJ", "qed"]
    
    # Salvar gráficos
    prop_dir = os.path.join(results_dir, 'mlp_qm9', f'att{att_index}_{property_name.replace(" ", "_")[:20]}')
    os.makedirs(prop_dir, exist_ok=True)
    
    # Gráficos SHAP e LIME
    horizontal_bar_plot(values=shap_global, feature_names=feature_names,
                       title=f"SHAP Global - {property_name}", save_path=prop_dir,
                       filename='shap_global.png', max_features=len(feature_names))
    plt.close()
    
    horizontal_bar_plot(values=shap_local, feature_names=feature_names,
                       title=f"SHAP Local - {property_name}", save_path=prop_dir,
                       filename='shap_local.png', max_features=len(feature_names))
    plt.close()
    
    horizontal_bar_plot(values=lime_local, feature_names=feature_names,
                       title=f"LIME Local - {property_name}", save_path=prop_dir,
                       filename='lime_local.png', max_features=len(feature_names))
    plt.close()
    
    # Comparação SHAP vs LIME
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes[0].barh(range(len(shap_local)), shap_local, color='steelblue')
    axes[0].set_yticks(range(len(feature_names)))
    axes[0].set_yticklabels(feature_names)
    axes[0].set_title('SHAP Local')
    axes[0].set_xlabel('Importância')
    
    axes[1].barh(range(len(lime_local)), lime_local, color='coral')
    axes[1].set_yticks(range(len(feature_names)))
    axes[1].set_yticklabels(feature_names)
    axes[1].set_title('LIME Local')
    axes[1].set_xlabel('Importância')
    
    plt.suptitle(f'Comparação SHAP vs LIME - {property_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(prop_dir, 'shap_vs_lime_comparison.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Curvas de treinamento
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(train_losses, label='Train Loss', color='blue')
    ax.plot(val_losses, label='Validation Loss', color='orange')
    ax.axhline(y=best_val_loss, color='green', linestyle='--', label=f'Best Val: {best_val_loss:.4f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MAE)')
    ax.set_title(f'Curvas de Treinamento (Otimizado) - {property_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.savefig(os.path.join(prop_dir, 'training_curves.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # Visualizar arquitetura
    visualize_mlp_architecture(model, input_dim, output_dim,
                               save_path=os.path.join(prop_dir, 'architecture.png'),
                               title=f'MLP Architecture (Optimized) - {property_name}')
    
    # Métricas de fidelidade
    print(f"\nCalculando métricas de fidelidade...")
    test_x_tensor = test_tensor.to(device)
    test_y_tensor = test_targets[:len(test_tensor)]
    
    with torch.no_grad():
        test_pred = model(test_x_tensor).cpu()
    
    analyzer = TabularAnalyzer(
        model=model, explainer=shap_explainer, explanation=np.array(shap_global),
        data=test_x_tensor.cpu().numpy(), y_true=test_y_tensor.numpy(),
        y_pred=test_pred.numpy(), device=device
    )
    
    fidelity = analyzer.get_metrics(classification=False)
    print(f"Fidelidade (pos, neg): {fidelity}")
    
    results = {
        'property_name': property_name,
        'att_index': att_index,
        'test_mae': test_mae,
        'best_val_mae': best_val_loss,
        'fidelity': fidelity,
        'shap_global': shap_global,
        'shap_local': shap_local,
        'lime_local': lime_local,
        'hyperparameters': best_params,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'model': model,
        'input_dim': input_dim
    }
    
    print(f"Resultados salvos em: {prop_dir}")
    return results

### 1.4 Executar Treinamento e Explicação para Todas as Propriedades do QM9

**Nota:** Este processo pode demorar. Para testes rápidos, pode-se selecionar apenas algumas propriedades.

In [ ]:
# Executar para todas as propriedades usando os hiperparâmetros otimizados
# Para teste rápido, use: selected_properties = [5, 6, 7]  # HOMO, LUMO, Gap
selected_properties = list(range(15))  # Todas as 15 propriedades

# Armazenar todos os resultados
all_results = {}

print(f"\n{'='*70}")
print("TREINAMENTO COM HIPERPARÂMETROS OTIMIZADOS PELO OPTUNA")
print(f"{'='*70}")
print(f"Melhores hiperparâmetros encontrados: {mlp_best_params}")

for att_idx in selected_properties:
    prop_name = property_names[att_idx]
    try:
        result = train_and_explain_mlp_optimized(
            qm9=qm9,
            att_index=att_idx,
            property_name=prop_name,
            device=device,
            results_dir=results_dir,
            best_params=mlp_best_params,  # Usar hiperparâmetros otimizados
            epochs=50,  # Mais épocas para treinamento final
            n_samples_explain=100
        )
        all_results[att_idx] = result
    except Exception as e:
        print(f"Erro ao processar propriedade {att_idx} ({prop_name}): {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*60}")
print("TREINAMENTO CONCLUÍDO")
print(f"{'='*60}")

In [ ]:
# Criar tabela resumo dos resultados com hiperparâmetros otimizados
import pandas as pd

summary_data = []
for att_idx, result in all_results.items():
    fid_pos, fid_neg = result['fidelity']
    summary_data.append({
        'Propriedade': result['property_name'],
        'MAE (Test)': f"{result['test_mae']:.4f}",
        'MAE (Val)': f"{result['best_val_mae']:.4f}",
        'Fidelidade+': f"{fid_pos.item():.4f}",
        'Fidelidade-': f"{fid_neg.item():.4f}",
        'Top SHAP Feature': ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
                            "NumRotatableBonds", "NumAromaticRings", "BalabanJ", "qed"][
                                np.argmax(np.abs(result['shap_global']))]
    })

df_summary = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("RESUMO DOS MODELOS MLP OTIMIZADOS PARA QM9")
print("="*80)
print(f"\nHiperparâmetros otimizados pelo Optuna:")
for k, v in mlp_best_params.items():
    print(f"  {k}: {v}")
print("\n")
print(df_summary.to_string(index=False))

# Salvar resumo em CSV
df_summary.to_csv(os.path.join(results_dir, 'mlp_qm9', 'summary_results_optimized.csv'), index=False)

# Salvar hiperparâmetros em JSON
import json
with open(os.path.join(results_dir, 'mlp_qm9', 'best_hyperparameters.json'), 'w') as f:
    json.dump(mlp_best_params, f, indent=2)

print(f"\nResumo salvo em: {os.path.join(results_dir, 'mlp_qm9', 'summary_results_optimized.csv')}")
print(f"Hiperparâmetros salvos em: {os.path.join(results_dir, 'mlp_qm9', 'best_hyperparameters.json')}")

### 1.5 Visualização Comparativa Global das Explicações

In [ ]:
# Criar heatmap de importâncias SHAP para todas as propriedades
feature_names = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
                "NumRotatableBonds", "NumAromaticRings", "BalabanJ", "qed"]

if len(all_results) > 0:
    # Matriz de importâncias SHAP
    shap_matrix = np.array([np.abs(result['shap_global']) for result in all_results.values()])
    prop_labels = [result['property_name'][:25] for result in all_results.values()]
    
    fig, ax = plt.subplots(figsize=(14, 10))
    im = ax.imshow(shap_matrix, cmap='YlOrRd', aspect='auto')
    
    ax.set_xticks(range(len(feature_names)))
    ax.set_xticklabels(feature_names, rotation=45, ha='right')
    ax.set_yticks(range(len(prop_labels)))
    ax.set_yticklabels(prop_labels)
    
    # Adicionar colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('|SHAP Value|', rotation=270, labelpad=20)
    
    ax.set_title('Heatmap de Importância SHAP Global\npara Todas as Propriedades QM9', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Features Physicochemical')
    ax.set_ylabel('Propriedade QM9')
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'mlp_qm9', 'shap_heatmap_all_properties.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Nenhum resultado disponível para visualização.")

---
## Parte 2: GCN + PCQM4 para Previsão HOMO-LUMO

### 2.1 Função para Visualizar Arquitetura da GCN

In [ ]:
def visualize_gcn_architecture(model, num_features, save_path=None, title="GCN Architecture"):
    """
    Cria uma visualização simplificada da arquitetura GCN.
    
    Parameters:
    -----------
    model : GCN
        Modelo GCN a ser visualizado
    num_features : int
        Número de features de entrada
    save_path : str, optional
        Caminho para salvar a imagem
    title : str
        Título do gráfico
    """
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 10)
    ax.axis('off')
    
    # Posições dos componentes
    components = [
        ('Input\nFeatures', 1, 5, num_features, 'lightblue'),
        ('GCNConv1\n+ BN + ReLU', 3, 5, 256, 'lightgreen'),
        ('GCNConv2\n+ BN + ReLU', 5, 5, 256, 'lightgreen'),
        ('GCNConv3\n+ BN + ReLU', 7, 5, 256, 'lightgreen'),
        ('Global\nPooling', 8.5, 5, 256, 'lightyellow'),
        ('Linear1\n+ Dropout', 10, 5, 256, 'lightcoral'),
        ('Output', 11.5, 5, 1, 'plum')
    ]
    
    for i, (name, x, y, dim, color) in enumerate(components):
        # Desenhar caixa
        height = min(3, 0.5 + dim/100)
        width = 1.2
        rect = plt.Rectangle((x - width/2, y - height/2), width, height, 
                             facecolor=color, edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        # Texto
        ax.text(x, y, name, ha='center', va='center', fontsize=9, fontweight='bold')
        ax.text(x, y - height/2 - 0.3, f'dim={dim}', ha='center', va='top', fontsize=8)
        
        # Setas de conexão
        if i < len(components) - 1:
            next_x = components[i+1][1]
            ax.annotate('', xy=(next_x - 0.6, y), xytext=(x + 0.6, y),
                       arrowprops=dict(arrowstyle='->', color='gray', lw=2))
    
    # Adicionar representação de grafo de entrada
    ax.text(1, 7.5, '🔵 Node Features', ha='center', fontsize=10)
    ax.text(1, 7, '➖ Edges', ha='center', fontsize=10)
    
    # Título e informações
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Informações do modelo
    total_params = sum(p.numel() for p in model.parameters())
    info_text = f"Total Parameters: {total_params:,}\nConvolution Layers: 3\nHidden Dim: 256\nDropout: 0.3"
    ax.text(6, 1.5, info_text, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Arquitetura GCN salva em: {save_path}")
    
    plt.show()
    return fig

### 2.2 Carregamento do Dataset PCQM4

In [ ]:
# Carregar dataset PCQM4
from torch_geometric.loader import DataLoader as GraphDataLoader

print("Carregando dataset PCQM4...")
graph_data = graph_datasets()

# Carregar dados do PCQM4 (contém HOMO-LUMO gap)
pcqm4_data = graph_data.prepare_data_graph(dataset_name='PCQM4')

# Obter número de features por nó
num_features = pcqm4_data[0].x.shape[1]

print(f"Dataset PCQM4 carregado!")
print(f"Número total de grafos: {len(pcqm4_data)}")
print(f"Número de features por nó: {num_features}")
print(f"Exemplo de grafo:")
print(f"  - Nós: {pcqm4_data[0].x.shape[0]}")
print(f"  - Arestas: {pcqm4_data[0].edge_index.shape[1]}")
print(f"  - Target (HOMO-LUMO gap): {pcqm4_data[0].y.item():.4f}")

### 2.3 Divisão dos Dados e Criação dos DataLoaders

In [ ]:
# Usar um subconjunto do PCQM4 para demonstração (dataset completo é muito grande)
# Para produção, usar mais dados
n_samples = 10000  # Usar 10k amostras para demonstração
subset_indices = list(range(n_samples))

# Criar subset
pcqm4_subset = [pcqm4_data[i] for i in subset_indices]

# Dividir em treino/validação/teste
train_size = int(0.8 * len(pcqm4_subset))
val_size = int(0.1 * len(pcqm4_subset))
test_size = len(pcqm4_subset) - train_size - val_size

train_data = pcqm4_subset[:train_size]
val_data = pcqm4_subset[train_size:train_size + val_size]
test_data = pcqm4_subset[train_size + val_size:]

# Criar DataLoaders
batch_size = 64
train_loader_gcn = GraphDataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader_gcn = GraphDataLoader(val_data, batch_size=batch_size, shuffle=False)
test_loader_gcn = GraphDataLoader(test_data, batch_size=batch_size, shuffle=False)

print(f"Dados divididos:")
print(f"  - Treino: {len(train_data)} grafos")
print(f"  - Validação: {len(val_data)} grafos")
print(f"  - Teste: {len(test_data)} grafos")

### 2.4 Otimização de Hiperparâmetros da GCN com Optuna

In [ ]:
from torch_geometric.nn import GCNConv, global_add_pool, global_mean_pool, global_max_pool
from torch.nn import Linear, BatchNorm1d, Dropout

# Modelo GCN flexível para otimização
class FlexibleGCN(torch.nn.Module):
    def __init__(self, num_features, hidden_dim=256, num_conv_layers=3, 
                 dropout_rate=0.3, pooling='add'):
        super(FlexibleGCN, self).__init__()
        
        self.num_conv_layers = num_conv_layers
        self.dropout_rate = dropout_rate
        
        # Primeira camada convolucional
        self.conv_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        
        self.conv_layers.append(GCNConv(num_features, hidden_dim))
        self.bn_layers.append(BatchNorm1d(hidden_dim))
        
        # Camadas convolucionais adicionais
        for _ in range(num_conv_layers - 1):
            self.conv_layers.append(GCNConv(hidden_dim, hidden_dim))
            self.bn_layers.append(BatchNorm1d(hidden_dim))
        
        # Pooling
        if pooling == 'add':
            self.pool = global_add_pool
        elif pooling == 'mean':
            self.pool = global_mean_pool
        else:
            self.pool = global_max_pool
        
        # Camadas lineares
        self.lin1 = Linear(hidden_dim, hidden_dim)
        self.lin2 = Linear(hidden_dim, 1)
        self.dropout = Dropout(dropout_rate)

    def forward(self, x, edge_index, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        
        # Camadas convolucionais
        for conv, bn in zip(self.conv_layers, self.bn_layers):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
        
        # Pooling global
        x = self.pool(x, batch)
        
        # MLP final
        x = self.dropout(F.relu(self.lin1(x)))
        x = self.lin2(x)
        
        return x


def create_gcn_objective(train_loader, val_loader, num_features, device):
    """
    Cria a função objetivo para otimização de hiperparâmetros da GCN com Optuna.
    """
    def objective(trial):
        # Sugerir hiperparâmetros
        hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256, 512])
        num_conv_layers = trial.suggest_int('num_conv_layers', 2, 5)
        dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
        pooling = trial.suggest_categorical('pooling', ['add', 'mean', 'max'])
        lr = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
        
        # Criar modelo
        model = FlexibleGCN(
            num_features=num_features,
            hidden_dim=hidden_dim,
            num_conv_layers=num_conv_layers,
            dropout_rate=dropout_rate,
            pooling=pooling
        ).to(device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.L1Loss()
        
        # Treinar por algumas épocas
        n_epochs = 15
        for epoch in range(n_epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.batch)
                loss = criterion(out.squeeze(), batch.y)
                loss.backward()
                optimizer.step()
            
            # Validação
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(device)
                    out = model(batch.x, batch.edge_index, batch.batch)
                    val_loss += criterion(out.squeeze(), batch.y).item()
            
            val_loss = val_loss / len(val_loader)
            
            # Reportar valor intermediário para pruning
            trial.report(val_loss, epoch)
            
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
        
        return val_loss
    
    return objective


def optimize_gcn_hyperparameters(train_loader, val_loader, num_features, device, 
                                  n_trials=50, timeout=600):
    """
    Otimiza hiperparâmetros da GCN usando Optuna.
    """
    print(f"\n{'='*60}")
    print("Otimizando hiperparâmetros GCN para PCQM4")
    print(f"{'='*60}")
    
    # Criar estudo Optuna
    study = optuna.create_study(
        direction='minimize',
        study_name='gcn_pcqm4',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    )
    
    # Criar função objetivo
    objective = create_gcn_objective(train_loader, val_loader, num_features, device)
    
    # Otimizar
    study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=True)
    
    print(f"\nMelhores hiperparâmetros GCN encontrados:")
    print(f"  Validation MAE: {study.best_value:.4f}")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study, study.best_params

In [ ]:
# Executar otimização de hiperparâmetros da GCN
gcn_study, gcn_best_params = optimize_gcn_hyperparameters(
    train_loader=train_loader_gcn,
    val_loader=val_loader_gcn,
    num_features=num_features,
    device=device,
    n_trials=30,  # Aumentar para resultados melhores
    timeout=600   # 10 minutos
)

# Visualizar resultados da otimização GCN
fig_history_gcn = optuna.visualization.plot_optimization_history(gcn_study)
fig_history_gcn.write_image(os.path.join(results_dir, 'optuna_studies', 'gcn_optimization_history.png'))
fig_history_gcn.show()

fig_importance_gcn = optuna.visualization.plot_param_importances(gcn_study)
fig_importance_gcn.write_image(os.path.join(results_dir, 'optuna_studies', 'gcn_param_importances.png'))
fig_importance_gcn.show()

# Visualizar espaço de parâmetros
fig_contour = optuna.visualization.plot_contour(gcn_study, params=['hidden_dim', 'num_conv_layers'])
fig_contour.write_image(os.path.join(results_dir, 'optuna_studies', 'gcn_param_contour.png'))
fig_contour.show()

print(f"\nGráficos de otimização GCN salvos em: {os.path.join(results_dir, 'optuna_studies')}")

### 2.5 Treinamento da GCN com Melhores Hiperparâmetros

In [ ]:
# Criar modelo GCN com os melhores hiperparâmetros do Optuna
print("Criando GCN com os melhores hiperparâmetros encontrados pelo Optuna...")
print(f"Hiperparâmetros: {gcn_best_params}")

gcn_model = FlexibleGCN(
    num_features=num_features,
    hidden_dim=gcn_best_params['hidden_dim'],
    num_conv_layers=gcn_best_params['num_conv_layers'],
    dropout_rate=gcn_best_params['dropout_rate'],
    pooling=gcn_best_params['pooling']
).to(device)

# Configuração do treinamento com hiperparâmetros otimizados
optimizer_gcn = torch.optim.Adam(
    gcn_model.parameters(), 
    lr=gcn_best_params['learning_rate'],
    weight_decay=gcn_best_params['weight_decay']
)
criterion_gcn = nn.L1Loss()

# Treinamento completo
epochs_gcn = 50  # Mais épocas para treinamento final
train_losses_gcn = []
val_losses_gcn = []
best_val_loss = float('inf')
best_model_state = None

print(f"\nIniciando treinamento final da GCN ({epochs_gcn} épocas)...")
for epoch in range(epochs_gcn):
    gcn_model.train()
    epoch_loss = 0
    
    for batch in train_loader_gcn:
        batch = batch.to(device)
        
        optimizer_gcn.zero_grad()
        out = gcn_model(batch.x, batch.edge_index, batch.batch)
        loss = criterion_gcn(out.squeeze(), batch.y)
        loss.backward()
        optimizer_gcn.step()
        
        epoch_loss += loss.item()
    
    train_loss = epoch_loss / len(train_loader_gcn)
    train_losses_gcn.append(train_loss)
    
    # Validação
    gcn_model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader_gcn:
            batch = batch.to(device)
            out = gcn_model(batch.x, batch.edge_index, batch.batch)
            val_loss += criterion_gcn(out.squeeze(), batch.y).item()
    
    val_loss = val_loss / len(val_loader_gcn)
    val_losses_gcn.append(val_loss)
    
    # Salvar melhor modelo
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = gcn_model.state_dict().copy()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs_gcn}] - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

# Carregar melhor modelo
gcn_model.load_state_dict(best_model_state)

# Avaliar no teste
gcn_model.eval()
test_loss = 0
with torch.no_grad():
    for batch in test_loader_gcn:
        batch = batch.to(device)
        out = gcn_model(batch.x, batch.edge_index, batch.batch)
        test_loss += criterion_gcn(out.squeeze(), batch.y).item()

test_mae_gcn = test_loss / len(test_loader_gcn)
print(f"\n✅ MAE no conjunto de teste (modelo otimizado): {test_mae_gcn:.4f}")
print(f"   Melhor MAE de validação: {best_val_loss:.4f}")

In [ ]:
# Plotar curvas de treinamento da GCN
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_losses_gcn, label='Train Loss', color='blue', linewidth=2)
ax.plot(val_losses_gcn, label='Validation Loss', color='orange', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (MAE)', fontsize=12)
ax.set_title('Curvas de Treinamento - GCN PCQM4 (HOMO-LUMO Gap)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'gcn_pcqm4', 'gcn_training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

# Visualizar arquitetura da GCN
visualize_gcn_architecture(
    gcn_model, num_features,
    save_path=os.path.join(results_dir, 'gcn_pcqm4', 'gcn_architecture.png'),
    title='GCN Architecture - PCQM4 HOMO-LUMO Gap Prediction'
)

### 2.5 Explicações com GNNExplainer

In [ ]:
# Selecionar alguns grafos de teste para explicação
n_explain = 5  # Número de grafos para explicar
explain_graphs = test_data[:n_explain]

print(f"Gerando explicações GNNExplainer para {n_explain} grafos...")

gnn_explanations = []

for i, graph in enumerate(explain_graphs):
    print(f"\nExplicando grafo {i+1}/{n_explain}...")
    
    # Criar explainer GNNExplainer
    gnn_explainer = GNNExplain(
        model=gcn_model,
        device=device,
        data=graph,
        epochs=200,
        mode='regression',
        task_level='graph',
        return_type='raw'
    )
    
    # Gerar explicação
    node_mask, edge_mask, explanation = gnn_explainer.explain()
    
    gnn_explanations.append({
        'graph_idx': i,
        'node_mask': node_mask,
        'edge_mask': edge_mask,
        'explanation': explanation,
        'num_nodes': graph.x.shape[0],
        'num_edges': graph.edge_index.shape[1],
        'target': graph.y.item()
    })
    
    print(f"  - Nós: {graph.x.shape[0]}, Arestas: {graph.edge_index.shape[1]}")
    print(f"  - Target HOMO-LUMO gap: {graph.y.item():.4f}")

print("\nExplicações GNNExplainer concluídas!")

### 2.6 Explicações com GraphSHAP

In [ ]:
print(f"Gerando explicações GraphSHAP para {n_explain} grafos...")

graphshap_explanations = []

for i, graph in enumerate(explain_graphs):
    print(f"\nExplicando grafo {i+1}/{n_explain} com GraphSHAP...")
    
    # Criar explainer GraphSHAP
    graphshap_explainer = GraphShap(
        data=graph,
        model=gcn_model,
        device=device,
        gpu=(device.type == 'cuda')
    )
    
    # Gerar explicação
    shap_values = graphshap_explainer.explain(num_samples=50)
    
    graphshap_explanations.append({
        'graph_idx': i,
        'shap_values': shap_values,
        'num_features': len(shap_values),
        'target': graph.y.item()
    })
    
    print(f"  - Features explicadas: {len(shap_values)}")
    print(f"  - Top 3 features mais importantes: {np.argsort(np.abs(shap_values))[-3:][::-1]}")

print("\nExplicações GraphSHAP concluídas!")

### 2.7 Explicações com GraphLIME

In [ ]:
print(f"Gerando explicações GraphLIME para {n_explain} grafos...")

graphlime_explanations = []

for i, graph in enumerate(explain_graphs):
    print(f"\nExplicando grafo {i+1}/{n_explain} com GraphLIME...")
    
    # Criar explainer GraphLIME
    graphlime_explainer = GraphLIME(
        model=gcn_model,
        device=device,
        rho=0.1
    )
    
    # Gerar explicação
    lime_values = graphlime_explainer.explain(data=graph, num_samples=100)
    
    graphlime_explanations.append({
        'graph_idx': i,
        'lime_values': lime_values,
        'num_features': len(lime_values),
        'target': graph.y.item()
    })
    
    print(f"  - Features explicadas: {len(lime_values)}")
    print(f"  - Top 3 features mais importantes: {np.argsort(np.abs(lime_values))[-3:][::-1]}")

print("\nExplicações GraphLIME concluídas!")

### 2.8 Visualização das Explicações GNN

In [ ]:
# Visualizar explicações para cada grafo
for i in range(n_explain):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    graph_idx = i
    
    # GNNExplainer - Node Mask
    gnn_exp = gnn_explanations[i]
    node_mask = np.array(gnn_exp['node_mask'])
    
    # Plotar importância dos nós (média por feature)
    if len(node_mask.shape) > 1:
        node_importance = np.mean(np.abs(node_mask), axis=1) if node_mask.shape[0] > 1 else np.abs(node_mask).flatten()
    else:
        node_importance = np.abs(node_mask)
    
    axes[0].bar(range(len(node_importance)), node_importance, color='steelblue')
    axes[0].set_xlabel('Node Index')
    axes[0].set_ylabel('Importance')
    axes[0].set_title(f'GNNExplainer - Node Importance\nGraph {i+1}')
    axes[0].grid(True, alpha=0.3)
    
    # GraphSHAP
    shap_vals = np.array(graphshap_explanations[i]['shap_values'])
    horizontal_bar_plot(
        values=shap_vals,
        feature_names=[f'F{j}' for j in range(len(shap_vals))],
        title=f'GraphSHAP - Feature Importance\nGraph {i+1}',
        save_path=None,
        filename=None,
        figsize=(6, 5)
    )
    # Fechar figura criada pelo horizontal_bar_plot e usar subplot
    plt.close()
    
    axes[1].barh(range(len(shap_vals)), shap_vals, color='coral')
    axes[1].set_xlabel('SHAP Value')
    axes[1].set_ylabel('Feature Index')
    axes[1].set_title(f'GraphSHAP - Feature Importance\nGraph {i+1}')
    axes[1].grid(True, alpha=0.3)
    
    # GraphLIME
    lime_vals = np.array(graphlime_explanations[i]['lime_values'])
    axes[2].barh(range(len(lime_vals)), lime_vals, color='forestgreen')
    axes[2].set_xlabel('LIME Value')
    axes[2].set_ylabel('Feature Index')
    axes[2].set_title(f'GraphLIME - Feature Importance\nGraph {i+1}')
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Comparação de Explicadores GNN - Grafo {i+1}\nTarget HOMO-LUMO Gap: {explain_graphs[i].y.item():.4f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'gcn_pcqm4', f'gnn_explanations_graph_{i+1}.png'), 
                dpi=300, bbox_inches='tight')
    plt.show()

print(f"\nGráficos salvos em: {os.path.join(results_dir, 'gcn_pcqm4')}")

### 2.9 Comparação Agregada dos Métodos de Explicação

In [ ]:
# Calcular importância média das features através de todos os grafos explicados
avg_shap = np.mean([np.abs(exp['shap_values']) for exp in graphshap_explanations], axis=0)
avg_lime = np.mean([np.abs(exp['lime_values']) for exp in graphlime_explanations], axis=0)

# Normalizar para comparação
avg_shap_norm = avg_shap / np.max(avg_shap) if np.max(avg_shap) > 0 else avg_shap
avg_lime_norm = avg_lime / np.max(avg_lime) if np.max(avg_lime) > 0 else avg_lime

# Criar gráfico de comparação
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras lado a lado
x = np.arange(len(avg_shap_norm))
width = 0.35

axes[0].bar(x - width/2, avg_shap_norm, width, label='GraphSHAP', color='coral', alpha=0.8)
axes[0].bar(x + width/2, avg_lime_norm, width, label='GraphLIME', color='forestgreen', alpha=0.8)
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Normalized Importance')
axes[0].set_title('Comparação GraphSHAP vs GraphLIME\n(Média de Importância Normalizada)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter plot para correlação
axes[1].scatter(avg_shap_norm, avg_lime_norm, alpha=0.7, s=100, c='purple')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y=x')
axes[1].set_xlabel('GraphSHAP (Normalizado)')
axes[1].set_ylabel('GraphLIME (Normalizado)')
axes[1].set_title('Correlação entre Métodos de Explicação')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Calcular correlação
from scipy.stats import spearmanr, pearsonr
corr_pearson, _ = pearsonr(avg_shap_norm, avg_lime_norm)
corr_spearman, _ = spearmanr(avg_shap_norm, avg_lime_norm)
axes[1].text(0.05, 0.95, f'Pearson: {corr_pearson:.3f}\nSpearman: {corr_spearman:.3f}',
             transform=axes[1].transAxes, fontsize=11, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Análise Comparativa dos Métodos de Explicação GNN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'gcn_pcqm4', 'explainer_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nCorrelação entre GraphSHAP e GraphLIME:")
print(f"  Pearson: {corr_pearson:.4f}")
print(f"  Spearman: {corr_spearman:.4f}")

### 2.10 Resumo Final dos Resultados GCN

In [ ]:
# Resumo final dos resultados GCN
print("="*70)
print("RESUMO FINAL - GCN PCQM4 (HOMO-LUMO Gap Prediction)")
print("="*70)

print(f"\n🎯 HIPERPARÂMETROS OTIMIZADOS (Optuna):")
for key, value in gcn_best_params.items():
    print(f"  - {key}: {value}")

print(f"\n📊 MÉTRICAS DO MODELO:")
print(f"  - Arquitetura: FlexibleGCN com {gcn_best_params['num_conv_layers']} camadas convolucionais")
print(f"  - Hidden dim: {gcn_best_params['hidden_dim']}")
print(f"  - Pooling: {gcn_best_params['pooling']}")
print(f"  - Dropout: {gcn_best_params['dropout_rate']}")
print(f"  - MAE (teste): {test_mae_gcn:.4f}")
print(f"  - Melhor MAE (validação): {best_val_loss:.4f}")
print(f"  - Épocas treinadas: {epochs_gcn}")

print(f"\n🔍 EXPLICAÇÕES GERADAS:")
print(f"  - GNNExplainer: {len(gnn_explanations)} grafos explicados")
print(f"  - GraphSHAP: {len(graphshap_explanations)} grafos explicados")
print(f"  - GraphLIME: {len(graphlime_explanations)} grafos explicados")

print(f"\n📈 ANÁLISE DE CORRELAÇÃO ENTRE EXPLICADORES:")
print(f"  - Correlação Pearson (SHAP vs LIME): {corr_pearson:.4f}")
print(f"  - Correlação Spearman (SHAP vs LIME): {corr_spearman:.4f}")

# Identificar features mais importantes (média)
top_features_shap = np.argsort(avg_shap_norm)[-5:][::-1]
top_features_lime = np.argsort(avg_lime_norm)[-5:][::-1]

print(f"\n⭐ TOP 5 FEATURES MAIS IMPORTANTES:")
print(f"  GraphSHAP: {list(top_features_shap)}")
print(f"  GraphLIME: {list(top_features_lime)}")

print(f"\n📁 RESULTADOS SALVOS EM:")
print(f"  {os.path.join(results_dir, 'gcn_pcqm4')}")

# Salvar resumo em arquivo JSON
import json
summary_gcn = {
    'model': 'FlexibleGCN (Optuna Optimized)',
    'dataset': 'PCQM4',
    'task': 'HOMO-LUMO Gap Prediction',
    'hyperparameters': gcn_best_params,
    'test_mae': test_mae_gcn,
    'best_val_mae': best_val_loss,
    'epochs': epochs_gcn,
    'n_graphs_explained': n_explain,
    'correlation_shap_lime': {
        'pearson': float(corr_pearson),
        'spearman': float(corr_spearman)
    },
    'top_features_shap': top_features_shap.tolist(),
    'top_features_lime': top_features_lime.tolist()
}

with open(os.path.join(results_dir, 'gcn_pcqm4', 'summary_gcn_optimized.json'), 'w') as f:
    json.dump(summary_gcn, f, indent=2)
    
print(f"\n✅ Resumo salvo em: {os.path.join(results_dir, 'gcn_pcqm4', 'summary_gcn_optimized.json')}")

---

## Conclusão

Este notebook demonstrou o uso completo do **ChemXAI Toolbox** com **otimização de hiperparâmetros via Optuna**:

### Teste 1 - MLP + QM9 (Descritores Physicochemical)
- ✅ **Otimização Optuna**: Busca automática de hiperparâmetros (layers, learning rate, dropout)
- ✅ Treinamento de modelos MLP otimizados para predição das 15 propriedades do QM9
- ✅ Explicações locais e globais com SHAP
- ✅ Explicações locais com LIME
- ✅ Comparação visual SHAP vs LIME
- ✅ Métricas de fidelidade usando TabularAnalyzer
- ✅ Visualização da arquitetura do modelo
- ✅ Early stopping com seleção do melhor modelo

### Teste 2 - GCN + PCQM4 (HOMO-LUMO Gap)
- ✅ **Otimização Optuna**: Busca automática de hiperparâmetros (hidden_dim, num_conv_layers, pooling, dropout, lr, weight_decay)
- ✅ Modelo FlexibleGCN com arquitetura configurável
- ✅ Treinamento de modelo GCN otimizado para predição do gap HOMO-LUMO
- ✅ Explicações com GNNExplainer (node/edge masks)
- ✅ Explicações com GraphSHAP
- ✅ Explicações com GraphLIME
- ✅ Análise comparativa dos métodos de explicação
- ✅ Visualização da arquitetura do modelo

### Optuna - Visualizações Geradas
- 📊 Histórico de otimização (convergência)
- 📈 Importância dos hiperparâmetros
- 🗺️ Contour plots do espaço de parâmetros

### Resultados Salvos
Todos os gráficos e métricas foram salvos no diretório `explanation_results/`:
- `mlp_qm9/` - Resultados dos modelos MLP + hiperparâmetros otimizados
- `gcn_pcqm4/` - Resultados do modelo GCN + hiperparâmetros otimizados
- `optuna_studies/` - Gráficos de otimização do Optuna